In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


In [2]:
df = pd.read_csv("DateFruit_Dataset.csv")
df.head()

,AREA,PERIMETER,MAJOR_AXIS,MINOR_AXIS,ECCENTRICITY,EQDIASQ,SOLIDITY,CONVEX_AREA,EXTENT,ASPECT_RATIO,...,KurtosisRR,KurtosisRG,KurtosisRB,EntropyRR,EntropyRG,EntropyRB,ALLdaub4RR,ALLdaub4RG,ALLdaub4RB,Class
0,422163,2378.908,837.8484,645.6693,0.6373,733.1539,0.9947,424428,0.7831,1.2976,...,3.2370,2.9574,4.2287,-59191263232,-50714214400,-39922372608,58.7255,54.9554,47.8400,BERHI
1,338136,2085.144,723.8198,595.2073,0.5690,656.1464,0.9974,339014,0.7795,1.2161,...,2.6228,2.6350,3.1704,-34233065472,-37462601728,-31477794816,50.0259,52.8168,47.8315,BERHI
2,526843,2647.394,940.7379,715.3638,0.6494,819.0222,0.9962,528876,0.7657,1.3150,...,3.7516,3.8611,4.7192,-93948354560,-74738221056,-60311207936,65.4772,59.2860,51.9378,BERHI
3,416063,2351.210,827.9804,645.2988,0.6266,727.8378,0.9948,418255,0.7759,1.2831,...,5.0401,8.6136,8.2618,-32074307584,-32060925952,-29575010304,43.3900,44.1259,41.1882,BERHI
4,347562,2160.354,763.9877,582.8359,0.6465,665.2291,0.9908,350797,0.7569,1.3108,...,2.7016,2.9761,4.4146,-39980974080,-35980042240,-25593278464,52.7743,50.9080,42.6666,BERHI


In [3]:
X = df.drop("Class",axis = 1)
y = df["Class"]

from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y = le.fit_transform(y)

In [4]:
# scaled the data 
X_train,X_test,y_train,y_test = train_test_split(
    X,y,test_size = 0.2,random_state = 42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [5]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader ,TensorDataset

In [6]:
X_train_tensor = torch.tensor(X_train_scaled,dtype = torch.float32)
y_train_tensor = torch.tensor(y_train , dtype = torch.long)

X_test_tensor = torch.tensor(X_test_scaled,dtype = torch.float32)
y_test_tensor = torch.tensor(y_test , dtype = torch.long)



In [7]:
train_dataset = TensorDataset(X_train_tensor,y_train_tensor)
test_dataset = TensorDataset(X_test_tensor,y_test_tensor)

In [8]:
train_loader = DataLoader(train_dataset,batch_size = 32 ,shuffle = True)
test_loader = DataLoader(test_dataset,batch_size = 32)

In [9]:
class ANN(nn.Module):
    def __init__(self):
        super(ANN,self).__init__()
        
        self.model = nn.Sequential(
            # 1st hidden layer that have X.shape input and 64 output
            nn.Linear(X.shape[1],64),
            nn.ReLU(),

            # 2nd hidden layer that have 64 input and output
            nn.Linear(64,64),
            nn.ReLU(),

            # 3rd output layer that have 64 input and 7 final output
            nn.Linear(64,7)
        )
    def forward(self,x):
        return self.model(x)
        

In [10]:
model = ANN()

# loss and optim
criteria = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())


In [11]:
# trainin the model

epochs = 100

for epoch in range(epochs):
    model.train()

    running_loss = 0.0

    for xb,yb in train_loader:
        optimizer.zero_grad()
        
        outputs = model(xb)
        loss = criteria(outputs,yb)
        loss.backward()
        optimizer.step() #parameter update

        running_loss += loss.item()

    train_loss = running_loss / len(train_loader)

    print(f"epoch = {epoch}/{epochs} ,loss = {train_loss}")

epoch = 0/100 ,loss = 1.7266733646392822
epoch = 1/100 ,loss = 1.1409123902735503
epoch = 2/100 ,loss = 0.7244519941184832
epoch = 3/100 ,loss = 0.5496922601824221
epoch = 4/100 ,loss = 0.46340089777241583
epoch = 5/100 ,loss = 0.38655553887719696
epoch = 6/100 ,loss = 0.3410650608332261
epoch = 7/100 ,loss = 0.3010756982409436
epoch = 8/100 ,loss = 0.2808517058906348
epoch = 9/100 ,loss = 0.2489680440529533
epoch = 10/100 ,loss = 0.23515381890794504
epoch = 11/100 ,loss = 0.22425397550282272
epoch = 12/100 ,loss = 0.2201191776472589
epoch = 13/100 ,loss = 0.201577828468188
epoch = 14/100 ,loss = 0.1935773692701174
epoch = 15/100 ,loss = 0.1942903529042783
epoch = 16/100 ,loss = 0.17862465200216873
epoch = 17/100 ,loss = 0.1754449280383794
epoch = 18/100 ,loss = 0.16756015141373096
epoch = 19/100 ,loss = 0.15257965838131698
epoch = 20/100 ,loss = 0.15704186171617196
epoch = 21/100 ,loss = 0.14911608041628546
epoch = 22/100 ,loss = 0.14444190117975939
epoch = 23/100 ,loss = 0.1401380083

In [12]:
#Evaluation

model.eval()

total = 0
correct = 0

with torch.no_grad():
    for xb,yb in test_loader:
        outputs = model(xb)
        # max_val,predicted_val = torch.max(outputs)
        _,predicted_val = torch.max(outputs,1)

        correct += (predicted_val == yb).sum().item()
        total += yb.size(0) #actual sample in each batch

print("total vals: ",total)
print("correct vals: ",correct)
        
print("accuracy : ", correct/total)     

total vals:  180
correct vals:  170
accuracy :  0.9444444444444444
